In [4]:
#!/usr/bin/env python3
"""
YOLO vs Ground Truth Comparison - VERIFIED SLIDES ONLY

Compares YOLO predictions with pathologist-verified annotations (_PO.xml)
to identify:
1. True Positives: YOLO correct (kept by pathologist)
2. False Positives: YOLO wrong (removed by pathologist) → DEFINITE NEGATIVES
3. False Negatives: YOLO missed (added by pathologist)

ONLY processes patches from verified slides (those with _PO.xml files)
"""

import xml.etree.ElementTree as ET
from pathlib import Path
import json
import shutil
from tqdm import tqdm
import re

# ====================================================================
# CONFIGURATION
# ====================================================================

GT_XML_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/Verified_xml2_full"
YOLO_XML_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch1"
PATCHES_DIR = "/home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images"
OUTPUT_DIR = "/home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results2"

PATCH_SIZE = 512
OVERLAP_THRESHOLD = 0.3  # 30% overlap needed

# ====================================================================
# XML PARSING
# ====================================================================

def parse_aperio_xml(xml_path):
    """Extract rectangular annotations from Aperio XML"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []
    
    for region in root.findall('.//Region'):
        vertices = []
        for vertex in region.findall('.//Vertex'):
            x = int(vertex.get('X'))
            y = int(vertex.get('Y'))
            vertices.append((x, y))
        
        if len(vertices) >= 4:
            xs = [v[0] for v in vertices]
            ys = [v[1] for v in vertices]
            annotations.append({
                'x_min': min(xs),
                'y_min': min(ys),
                'x_max': max(xs),
                'y_max': max(ys),
                'id': region.get('Id')
            })
    
    return annotations


def parse_filename(filename):
    """Extract WSI ID and coordinates from patch filename"""
    match = re.search(r'(.+?)_x(\d+)_y(\d+)', filename.lower())
    if match:
        return match.group(1), int(match.group(2)), int(match.group(3))
    return filename.replace('.png', ''), 0, 0


# ====================================================================
# GEOMETRY
# ====================================================================

def get_patch_bounds(offset_x, offset_y, patch_size=512):
    """Get global coordinates of a patch"""
    return {
        'x_min': offset_x,
        'y_min': offset_y,
        'x_max': offset_x + patch_size,
        'y_max': offset_y + patch_size
    }


def calculate_overlap(bbox1, bbox2):
    """Calculate intersection area between two bounding boxes"""
    x_left = max(bbox1['x_min'], bbox2['x_min'])
    y_top = max(bbox1['y_min'], bbox2['y_min'])
    x_right = min(bbox1['x_max'], bbox2['x_max'])
    y_bottom = min(bbox1['y_max'], bbox2['y_max'])
    
    if x_right < x_left or y_bottom < y_top:
        return 0, 0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    bbox1_area = (bbox1['x_max'] - bbox1['x_min']) * (bbox1['y_max'] - bbox1['y_min'])
    
    return intersection_area, bbox1_area


def patch_contains_annotation(patch_bounds, annotation, threshold=0.3):
    """Check if patch contains significant overlap with annotation"""
    overlap_area, annot_area = calculate_overlap(annotation, patch_bounds)
    
    if annot_area == 0:
        return False
    
    overlap_fraction = overlap_area / annot_area
    return overlap_fraction >= threshold


# ====================================================================
# MAIN COMPARISON
# ====================================================================

def compare_verified_slides(gt_xml_dir, yolo_xml_dir, patches_dir, output_dir):
    """
    Compare YOLO predictions with pathologist verifications
    
    Process:
    1. Load verified annotations (_PO2.xml) - these are the GROUND TRUTH
    2. Load YOLO predictions (*.xml without _PO)
    3. For each patch from verified slides:
       - Check if YOLO detected bacteria
       - Check if pathologist confirmed (in _PO.xml)
       - Classify accordingly
    """
    
    print("\n" + "="*80)
    print("🔬 VERIFIED SLIDES: YOLO vs PATHOLOGIST COMPARISON")
    print("="*80)
    
    gt_dir = Path(gt_xml_dir)
    yolo_dir = Path(yolo_xml_dir)
    patches_path = Path(patches_dir)
    output_path = Path(output_dir)
    
    # Create output directories
    for subdir in ['true_positives', 'false_positives', 'false_negatives', 'reports']:
        (output_path / subdir).mkdir(parents=True, exist_ok=True)
    
    # Step 1: Load VERIFIED annotations (Ground Truth)
    print("\n📂 Loading VERIFIED annotations (_PO.xml)...")
    gt_xmls = {}
    verified_slides = []
    
    for xml_file in gt_dir.glob("*_PO2.xml"):
        wsi_id = xml_file.stem.replace('_PO2', '')
        gt_xmls[wsi_id] = parse_aperio_xml(xml_file)
        verified_slides.append(wsi_id)
        print(f"   ✓ {wsi_id}: {len(gt_xmls[wsi_id])} verified annotations")
    
    if not verified_slides:
        print("\n❌ ERROR: No verified slides found!")
        return
    
    print(f"\n✅ Found {len(verified_slides)} verified slides")
    print(f"   Slides: {sorted(verified_slides)}")
    
    # Step 2: Load YOLO predictions for verified slides
    print("\n📂 Loading YOLO predictions for verified slides...")
    yolo_xmls = {}
    
    for wsi_id in verified_slides:
        yolo_xml_path = yolo_dir / f"{wsi_id}.xml"
        if yolo_xml_path.exists():
            yolo_xmls[wsi_id] = parse_aperio_xml(yolo_xml_path)
            print(f"   ✓ {wsi_id}: {len(yolo_xmls[wsi_id])} YOLO detections")
        else:
            yolo_xmls[wsi_id] = []
            print(f"   ⚠️ {wsi_id}: No YOLO predictions found")
    
    # Step 3: Find all patches from verified slides
    print("\n🔍 Finding patches from verified slides...")
    all_patches = list(patches_path.glob("*.png"))
    verified_patches = []
    
    for patch_file in all_patches:
        wsi_id, _, _ = parse_filename(patch_file.name)
        if wsi_id in verified_slides:
            verified_patches.append(patch_file)
    
    print(f"   Total patches in dataset: {len(all_patches):,}")
    print(f"   Patches from verified slides: {len(verified_patches):,}")
    
    # Step 4: Classify patches
    print("\n🔬 Classifying patches...")
    
    classification = {
        'true_positive': [],   # YOLO correct (pathologist kept it)
        'false_positive': [],  # YOLO wrong (pathologist removed it) → DEFINITE NEGATIVES!
        'false_negative': []   # YOLO missed (pathologist added it)
    }
    
    for patch_file in tqdm(verified_patches, desc="Analyzing patches"):
        wsi_id, offset_x, offset_y = parse_filename(patch_file.name)
        patch_bounds = get_patch_bounds(offset_x, offset_y, PATCH_SIZE)
        
        # Check if GROUND TRUTH (pathologist) confirmed bacteria in this patch
        has_gt = False
        gt_annotations_in_patch = []
        for annot in gt_xmls[wsi_id]:
            if patch_contains_annotation(patch_bounds, annot, OVERLAP_THRESHOLD):
                has_gt = True
                gt_annotations_in_patch.append(annot)
        
        # Check if YOLO detected bacteria in this patch
        has_yolo = False
        yolo_detections_in_patch = []
        for annot in yolo_xmls[wsi_id]:
            if patch_contains_annotation(patch_bounds, annot, OVERLAP_THRESHOLD):
                has_yolo = True
                yolo_detections_in_patch.append(annot)
        
        # Classify
        patch_info = {
            'patch': patch_file.name,
            'wsi_id': wsi_id,
            'offset_x': offset_x,
            'offset_y': offset_y,
            'gt_count': len(gt_annotations_in_patch),
            'yolo_count': len(yolo_detections_in_patch)
        }
        
        if has_gt and has_yolo:
            # YOLO detected AND pathologist confirmed → TRUE POSITIVE
            classification['true_positive'].append(patch_info)
        elif not has_gt and has_yolo:
            # YOLO detected BUT pathologist removed → FALSE POSITIVE (DEFINITE NEGATIVE!)
            classification['false_positive'].append(patch_info)
        elif has_gt and not has_yolo:
            # YOLO missed BUT pathologist added → FALSE NEGATIVE
            classification['false_negative'].append(patch_info)
        # else: No GT and No YOLO → Skip (not informative for retraining)
    
    # Step 5: Report results
    print("\n" + "="*80)
    print("📊 CLASSIFICATION RESULTS")
    print("="*80)
    
    tp = len(classification['true_positive'])
    fp = len(classification['false_positive'])
    fn = len(classification['false_negative'])
    
    print(f"\n✅ TRUE POSITIVES (YOLO Correct):")
    print(f"   Count: {tp:,} patches")
    print(f"   → YOLO detected bacteria, pathologist confirmed")
    print(f"   → Use as POSITIVE training data for next iteration")
    
    print(f"\n⚠️  FALSE POSITIVES (YOLO Wrong):")
    print(f"   Count: {fp:,} patches")
    print(f"   → YOLO detected bacteria, pathologist REMOVED")
    print(f"   → ⭐ DEFINITE NEGATIVES - use for NEGATIVE training data!")
    
    print(f"\n❌ FALSE NEGATIVES (YOLO Missed):")
    print(f"   Count: {fn:,} patches")
    print(f"   → YOLO missed bacteria, pathologist ADDED")
    print(f"   → Model needs improvement to catch these")
    
    # Calculate metrics
    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0
    
    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0
    
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0
    
    print(f"\n📈 YOLO PERFORMANCE METRICS:")
    print(f"   Precision: {precision:.3f} ({precision*100:.1f}%)")
    print(f"   Recall:    {recall:.3f} ({recall*100:.1f}%)")
    print(f"   F1 Score:  {f1:.3f}")
    
    # Step 6: Save results
    print(f"\n💾 Saving results...")
    
    report = {
        'verified_slides': verified_slides,
        'total_verified_patches': len(verified_patches),
        'classification': {
            'true_positive': tp,
            'false_positive': fp,
            'false_negative': fn
        },
        'metrics': {
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
    }
    
    # Save summary
    report_path = output_path / 'reports' / 'verification_summary.json'
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"   ✓ Summary: {report_path}")
    
    # Save detailed patch lists
    for category, patches in classification.items():
        if len(patches) > 0:
            list_path = output_path / 'reports' / f'{category}_patches.json'
            with open(list_path, 'w') as f:
                json.dump(patches, f, indent=2)
            print(f"   ✓ {category}: {list_path}")
    
    # Save text report
    text_path = output_path / 'reports' / 'verification_report.txt'
    with open(text_path, 'w') as f:
        f.write("YOLO vs PATHOLOGIST VERIFICATION REPORT\n")
        f.write("="*80 + "\n\n")
        
        f.write("VERIFIED SLIDES:\n")
        for slide in sorted(verified_slides):
            f.write(f"  • {slide}\n")
        
        f.write(f"\nPATCHES ANALYZED: {len(verified_patches):,}\n\n")
        
        f.write("CLASSIFICATION:\n")
        f.write(f"  ✅ True Positives:  {tp:,} (YOLO correct)\n")
        f.write(f"  ⚠️  False Positives: {fp:,} (YOLO wrong → DEFINITE NEGATIVES)\n")
        f.write(f"  ❌ False Negatives: {fn:,} (YOLO missed)\n\n")
        
        f.write("METRICS:\n")
        f.write(f"  Precision: {precision:.3f}\n")
        f.write(f"  Recall:    {recall:.3f}\n")
        f.write(f"  F1 Score:  {f1:.3f}\n\n")
        
        f.write("RECOMMENDATIONS FOR NEXT TRAINING:\n")
        f.write(f"  1. Add {tp:,} TRUE POSITIVE patches to positive training set\n")
        f.write(f"  2. Add {fp:,} FALSE POSITIVE patches to negative training set (DEFINITE NEGATIVES)\n")
        f.write(f"  3. Review {fn:,} FALSE NEGATIVE patches - model needs improvement\n")
    
    print(f"   ✓ Text report: {text_path}")
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    print(f"\n🎯 KEY FINDINGS:")
    print(f"   • Analyzed {len(verified_patches):,} patches from {len(verified_slides)} verified slides")
    print(f"   • {tp:,} patches: YOLO correct (use as positives)")
    print(f"   • {fp:,} patches: YOLO wrong (use as DEFINITE negatives)")
    print(f"   • {fn:,} patches: YOLO missed (needs improvement)")
    
    print(f"\n📂 Results saved to: {output_path}")
    
    return classification, report


# ====================================================================
# MAIN
# ====================================================================

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print("🔬 YOLO VERIFICATION ANALYSIS")
    print("="*80)
    
    print("\n💡 Purpose:")
    print("   Compare YOLO predictions with pathologist verifications")
    print("   to identify:")
    print("   1. Correct detections (use as positive data)")
    print("   2. Wrong detections (use as DEFINITE negative data)")
    print("   3. Missed detections (model improvement needed)")
    
    proceed = input("\n▶️  Start verification analysis? (y/n): ").strip().lower()
    if proceed != 'y':
        print("Cancelled.")
        exit(0)
    
    classification, report = compare_verified_slides(
        GT_XML_DIR,
        YOLO_XML_DIR,
        PATCHES_DIR,
        OUTPUT_DIR
    )
    
    print("\n🎉 Done! Check the reports folder for detailed results.")


🔬 YOLO VERIFICATION ANALYSIS

💡 Purpose:
   Compare YOLO predictions with pathologist verifications
   to identify:
   1. Correct detections (use as positive data)
   2. Wrong detections (use as DEFINITE negative data)
   3. Missed detections (model improvement needed)



▶️  Start verification analysis? (y/n):  y



🔬 VERIFIED SLIDES: YOLO vs PATHOLOGIST COMPARISON

📂 Loading VERIFIED annotations (_PO.xml)...
   ✓ 593450: 7 verified annotations
   ✓ 593444: 2 verified annotations
   ✓ 593449: 48 verified annotations
   ✓ 593448: 0 verified annotations
   ✓ 593453: 0 verified annotations
   ✓ 593440: 13 verified annotations
   ✓ 593447: 2 verified annotations
   ✓ 593441: 0 verified annotations
   ✓ 593438: 21 verified annotations
   ✓ 593454: 2 verified annotations
   ✓ 593434: 0 verified annotations
   ✓ 593452: 34 verified annotations
   ✓ 593446: 6 verified annotations
   ✓ 593439: 3 verified annotations
   ✓ 593436: 31 verified annotations
   ✓ 593437: 17 verified annotations
   ✓ 593451: 3 verified annotations
   ✓ 593433: 0 verified annotations
   ✓ 593445: 12 verified annotations
   ✓ 593435: 15 verified annotations

✅ Found 20 verified slides
   Slides: ['593433', '593434', '593435', '593436', '593437', '593438', '593439', '593440', '593441', '593444', '593445', '593446', '593447', '59344

Analyzing patches: 100%|█████████████| 106663/106663 [00:04<00:00, 22570.96it/s]



📊 CLASSIFICATION RESULTS

✅ TRUE POSITIVES (YOLO Correct):
   Count: 262 patches
   → YOLO detected bacteria, pathologist confirmed
   → Use as POSITIVE training data for next iteration

⚠️  FALSE POSITIVES (YOLO Wrong):
   Count: 299 patches
   → YOLO detected bacteria, pathologist REMOVED
   → ⭐ DEFINITE NEGATIVES - use for NEGATIVE training data!

❌ FALSE NEGATIVES (YOLO Missed):
   Count: 0 patches
   → YOLO missed bacteria, pathologist ADDED
   → Model needs improvement to catch these

📈 YOLO PERFORMANCE METRICS:
   Precision: 0.467 (46.7%)
   Recall:    1.000 (100.0%)
   F1 Score:  0.637

💾 Saving results...
   ✓ Summary: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results2/reports/verification_summary.json
   ✓ true_positive: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch/verified_comparison_results2/reports/true_positive_patches.json
   ✓ false_positive: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_bat

In [8]:
import shutil
import json
import random
import yaml
import xml.etree.ElementTree as ET
import numpy as np
from pathlib import Path
from typing import Dict, List, Tuple
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from ultralytics import YOLO
from dataclasses import dataclass
from collections import defaultdict

# ====================================================================
# CONFIGURATION (FIXED PATHS)
# ====================================================================
@dataclass
class Config:
    PROJECT_ROOT: Path = Path("/home/biopsy_gregorova/hpylori_project")
    BASE_DIR: Path = PROJECT_ROOT / "master-data/separated_patches"
    OUTPUT_DIR: Path = PROJECT_ROOT / "yolo_it2_final"
    
    # Ground Truth Data
    ORIGINAL_POS_IMG: Path = BASE_DIR / "positive_new/images"
    ORIGINAL_POS_LBL: Path = BASE_DIR / "positive_new/labels"
    ORIGINAL_NEG_IMG: Path = BASE_DIR / "negative/images"
    
    # Verified Iterations (JSON Paths)
    VERIF_IT1_BASE: Path = PROJECT_ROOT / "wsi_global_xmls_test_batch/verified_comparison_results1"
    VERIF_IT2_BASE: Path = PROJECT_ROOT / "wsi_global_xmls_test_batch/verified_comparison_results2"
    
    # XML Sources (Explicitly Separated)
    XML_SOURCE_IT1: Path = PROJECT_ROOT / "wsi_global_xmls_test_batch/Verified_xml1_full"
    XML_SOURCE_IT2: Path = PROJECT_ROOT / "wsi_global_xmls_test_batch/Verified_xml2_full"
    
    # Image Sources
    # FIX: Added 'separated_patches' to IT1 path so it finds the images!
    IT1_IMG_SOURCE: Path = BASE_DIR / "test_data_full/images" 
    IT2_IMG_SOURCE: Path = BASE_DIR / "test_data_full/images"
    
    # Model
    PRETRAINED_MODEL: Path = PROJECT_ROOT / "yolo_optimal_active1/train_20260109_131000/weights/best.pt"
    
    # Training Parameters
    IMG_SIZE: int = 512
    BATCH_SIZE: int = 16
    EPOCHS: int = 150
    PATIENCE: int = 50
    NEG_TO_POS_RATIO: float = 0.25
    VAL_SIZE: float = 0.20
    SEED: int = 42
    WORKERS: int = 4
    
    # Detection Thresholds
    MIN_OVERLAP_RATIO: float = 0.0
    MIN_BOX_SIZE: int = 2 
    
    HYPS: Dict = None
    
    def __post_init__(self):
        self.HYPS = {
            'lr0': 0.001, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 
            'box': 7.5, 'cls': 1.0, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 
            'hsv_v': 0.4, 'degrees': 180, 'translate': 0.1, 'scale': 0.1, 
            'flipud': 0.5, 'fliplr': 0.5, 'mosaic': 0.0, 'mixup': 0.0,
        }

# ====================================================================
# STATISTICS TRACKER
# ====================================================================
class StatsTracker:
    def __init__(self):
        self.stats = defaultdict(int)
        self.failed_samples = []
        
    def increment(self, key: str, count: int = 1):
        self.stats[key] += count
        
    def log_failure(self, reason: str, details: Dict):
        self.failed_samples.append({'reason': reason, **details})
        
    def report(self):
        print("\n" + "="*70)
        print("📊 DATASET STATISTICS REPORT")
        print("="*70)
        print("\n🟢 POSITIVE SAMPLES:")
        print(f"   Ground Truth (Original):     {self.stats['pos_original']:4d}")
        print(f"   Verified IT1 Recovered:      {self.stats['pos_it1']:4d}")
        print(f"   Verified IT2 Recovered:      {self.stats['pos_it2']:4d}")
        print(f"   {'─'*40}")
        print(f"   Total Positives:             {self.stats['pos_total']:4d}")
        print("\n🔴 NEGATIVE SAMPLES:")
        print(f"   Hard Negatives (FP IT2):     {self.stats['neg_hard_it2']:4d}")
        print(f"   Hard Negatives (FP IT1):     {self.stats['neg_hard_it1']:4d}")
        print(f"   Easy Negatives (Original):   {self.stats['neg_easy']:4d}")
        print(f"   {'─'*40}")
        print(f"   Total Negatives:             {self.stats['neg_total']:4d}")
        ratio = self.stats['neg_total'] / max(self.stats['pos_total'], 1)
        print(f"\n⚖️  Negative:Positive Ratio:     {ratio:.3f}")
        
        if self.stats['fail_xml'] or self.stats['fail_overlap']:
            print("\n⚠️  RECOVERY ISSUES:")
            print(f"   Missing XML Files:           {self.stats['fail_xml']:4d}")
            print(f"   No Overlap (Below Threshold):{self.stats['fail_overlap']:4d}")
        print("="*70 + "\n")

# ====================================================================
# DATA MANAGER
# ====================================================================
class DataManager:
    def __init__(self, config: Config):
        self.cfg = config
        self.used_patch_names = set()
        self.xml_cache = {} 
        self.stats = StatsTracker()
        
    def load_json(self, path: Path) -> List[Dict]:
        if not path.exists(): return []
        with open(path, 'r') as f: return json.load(f)
    
    def parse_xml_for_wsi(self, wsi_id: str, xml_dir: Path, suffix: str) -> List[Dict]:
        """Parse XML for a specific version."""
        cache_key = f"{wsi_id}_{suffix}"
        if cache_key in self.xml_cache:
            return self.xml_cache[cache_key]
        
        # Priority list for filenames
        candidates = [
            xml_dir / f"{wsi_id}{suffix}",  # e.g. 593449_PO.xml
            xml_dir / f"{wsi_id}.xml"       # Fallback
        ]
        
        root = None
        for xml_path in candidates:
            if xml_path.exists():
                try:
                    tree = ET.parse(xml_path)
                    root = tree.getroot()
                    break 
                except ET.ParseError: pass
        
        annotations = []
        if root:
            for region in root.findall('.//Region'):
                vertices = [(int(v.get('X')), int(v.get('Y'))) for v in region.findall('.//Vertex')]
                if len(vertices) >= 2:
                    xs, ys = zip(*vertices)
                    annotations.append({
                        'x_min': min(xs), 'y_min': min(ys),
                        'x_max': max(xs), 'y_max': max(ys)
                    })
        self.xml_cache[cache_key] = annotations
        return annotations
    
    def get_labels_for_patch(self, wsi_id: str, offset_x: int, offset_y: int, 
                            xml_dir: Path, xml_suffix: str) -> Tuple[List[str], bool]:
        """Generate labels using the specific XML version"""
        annotations = self.parse_xml_for_wsi(wsi_id, xml_dir, xml_suffix)
        
        if not annotations:
            return [], False
        
        yolo_labels = []
        patch_rect = [offset_x, offset_y, offset_x + self.cfg.IMG_SIZE, offset_y + self.cfg.IMG_SIZE]
        has_overlap = False
        
        for ann in annotations:
            # Loose Match: Check for ANY intersection
            x_left = max(patch_rect[0], ann['x_min'])
            y_top = max(patch_rect[1], ann['y_min'])
            x_right = min(patch_rect[2], ann['x_max'])
            y_bottom = min(patch_rect[3], ann['y_max'])
            
            if x_right > x_left and y_bottom > y_top:
                has_overlap = True
                
                box_x_min = max(0, ann['x_min'] - offset_x)
                box_y_min = max(0, ann['y_min'] - offset_y)
                box_x_max = min(self.cfg.IMG_SIZE, ann['x_max'] - offset_x)
                box_y_max = min(self.cfg.IMG_SIZE, ann['y_max'] - offset_y)
                
                w = box_x_max - box_x_min
                h = box_y_max - box_y_min
                
                if w >= self.cfg.MIN_BOX_SIZE and h >= self.cfg.MIN_BOX_SIZE:
                    cx = (box_x_min + w / 2) / self.cfg.IMG_SIZE
                    cy = (box_y_min + h / 2) / self.cfg.IMG_SIZE
                    nw = w / self.cfg.IMG_SIZE
                    nh = h / self.cfg.IMG_SIZE
                    yolo_labels.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
                    
        return yolo_labels, has_overlap
    
    def collect_positives(self) -> List[Dict]:
        samples = []
        print("\n🔍 Collecting Positive Samples...")
        
        # 1. Ground Truth
        print("   📁 Loading Ground Truth (Original)...")
        for img in self.cfg.ORIGINAL_POS_IMG.glob("*.png"):
            lbl = self.cfg.ORIGINAL_POS_LBL / f"{img.stem}.txt"
            if lbl.exists():
                samples.append({'img': img, 'lbl_type': 'file', 'src': lbl, 'source': 'ground_truth'})
                self.used_patch_names.add(img.name)
                self.stats.increment('pos_original')
        
        # 2. Verified Positives (EXPLICIT MAPPING)
        # Format: (JSON Path, Image Dir, XML Dir, XML Suffix, Stat Key)
        json_sources = [
            (self.cfg.VERIF_IT1_BASE / "reports/true_positive_patches.json", 
             self.cfg.IT1_IMG_SOURCE, self.cfg.XML_SOURCE_IT1, '_PO.xml', 'pos_it1'),
             
            (self.cfg.VERIF_IT2_BASE / "reports/true_positive_patches.json", 
             self.cfg.IT2_IMG_SOURCE, self.cfg.XML_SOURCE_IT2, '_PO2.xml', 'pos_it2')
        ]
        
        for json_path, img_root, xml_dir, suffix, stat_key in json_sources:
            print(f"   📁 Loading {stat_key} from {json_path.parent.parent.name}...")
            items = self.load_json(json_path)
            
            for item in tqdm(items, desc=f"   Processing {stat_key}", leave=False):
                img_path = img_root / item['patch']
                # Debug print for first failure
                if not img_path.exists():
                    if self.stats.stats[f'{stat_key}_missing_img'] == 0:
                        print(f"⚠️  DEBUG: Image not found at {img_path}")
                    self.stats.increment(f'{stat_key}_missing_img')
                    continue
                
                labels, has_overlap = self.get_labels_for_patch(
                    item.get('wsi_id'), item.get('offset_x'), item.get('offset_y'), 
                    xml_dir, suffix
                )
                
                if has_overlap and labels:
                    samples.append({
                        'img': img_path, 'lbl_type': 'generated', 'labels': labels, 'source': stat_key
                    })
                    self.used_patch_names.add(img_path.name)
                    self.stats.increment(stat_key)
                else:
                    self.stats.increment('fail_overlap')
        
        self.stats.stats['pos_total'] = len(samples)
        print(f"   ✅ Total Positive Samples: {len(samples)}")
        return samples
    
    def collect_negatives(self, target_count: int) -> List[Dict]:
        samples = []
        print(f"\n🔍 Collecting {target_count} Negative Samples...")
        
        hard_sources = [
            (self.cfg.VERIF_IT2_BASE / "reports/false_positive_patches.json", self.cfg.IT2_IMG_SOURCE, 'neg_hard_it2'),
            (self.cfg.VERIF_IT1_BASE / "reports/false_positive_patches.json", self.cfg.IT1_IMG_SOURCE, 'neg_hard_it1')
        ]
        
        for json_path, img_root, stat_key in hard_sources:
            items = self.load_json(json_path)
            for item in items:
                img_path = img_root / item['patch']
                if img_path.exists() and img_path.name not in self.used_patch_names:
                    samples.append({'img': img_path, 'lbl_type': 'empty', 'source': stat_key})
                    self.used_patch_names.add(img_path.name)
                    self.stats.increment(stat_key)
                    if len(samples) >= target_count: break
            if len(samples) >= target_count: break
        
        remaining = target_count - len(samples)
        if remaining > 0:
            print(f"   📁 Adding {remaining} Easy Negatives...")
            easy_negs = [x for x in self.cfg.ORIGINAL_NEG_IMG.glob("*.png") if x.name not in self.used_patch_names]
            random.shuffle(easy_negs)
            for img in easy_negs[:remaining]:
                samples.append({'img': img, 'lbl_type': 'empty', 'source': 'neg_easy'})
                self.used_patch_names.add(img.name)
                self.stats.increment('neg_easy')
        
        self.stats.stats['neg_total'] = len(samples)
        return samples

    def prepare_dataset(self):
        positives = self.collect_positives()
        n_pos = len(positives)
        n_neg = max(int(n_pos * self.cfg.NEG_TO_POS_RATIO), 100)
        negatives = self.collect_negatives(n_neg)
        
        self.stats.report()
        
        dataset = positives + negatives
        random.shuffle(dataset)
        train_data, val_data = train_test_split(dataset, test_size=self.cfg.VAL_SIZE, random_state=self.cfg.SEED)
        
        if self.cfg.OUTPUT_DIR.exists(): shutil.rmtree(self.cfg.OUTPUT_DIR)
        self._write_split(train_data, 'train')
        self._write_split(val_data, 'val')
        self._create_yaml()
        
        return self.cfg.OUTPUT_DIR

    def _write_split(self, dataset, split):
        img_dir = self.cfg.OUTPUT_DIR / "images" / split
        lbl_dir = self.cfg.OUTPUT_DIR / "labels" / split
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        
        for data in tqdm(dataset, desc=f"✍️  Writing {split}"):
            shutil.copy2(data['img'], img_dir / data['img'].name)
            lbl_file = lbl_dir / f"{data['img'].stem}.txt"
            if data['lbl_type'] == 'file': shutil.copy2(data['src'], lbl_file)
            elif data['lbl_type'] == 'generated':
                with open(lbl_file, 'w') as f: f.write("\n".join(data['labels']))
            else: lbl_file.touch()

    def _create_yaml(self):
        yaml_data = {'path': str(self.cfg.OUTPUT_DIR.absolute()), 'train': 'images/train', 'val': 'images/val', 'names': {0: 'H_pylori'}}
        with open(self.cfg.OUTPUT_DIR / 'data.yaml', 'w') as f: yaml.dump(yaml_data, f)

def main():
    random.seed(42)
    np.random.seed(42)
    print("\n🚀 H. PYLORI DETECTION - ITERATION 2 (FIXED PATHS)")
    
    config = Config()
    dm = DataManager(config)
    dataset_dir = dm.prepare_dataset()
    
    print("\n🔥 STARTING TRAINING")
    model = YOLO(str(config.PRETRAINED_MODEL))
    model.train(
        data=str(dataset_dir / 'data.yaml'),
        epochs=config.EPOCHS,
        patience=config.PATIENCE,
        batch=config.BATCH_SIZE,
        imgsz=config.IMG_SIZE,
        project=str(config.OUTPUT_DIR),
        name='train',
        workers=config.WORKERS,
        resume=False,
        **config.HYPS
    )

if __name__ == "__main__":
    main()


🚀 H. PYLORI DETECTION - ITERATION 2 (FIXED PATHS)

🔍 Collecting Positive Samples...
   📁 Loading Ground Truth (Original)...
   📁 Loading pos_it1 from verified_comparison_results1...


   📁 Loading pos_it2 from verified_comparison_results2...


   ✅ Total Positive Samples: 2412

🔍 Collecting 603 Negative Samples...

📊 DATASET STATISTICS REPORT

🟢 POSITIVE SAMPLES:
   Ground Truth (Original):     1153
   Verified IT1 Recovered:       997
   Verified IT2 Recovered:       262
   ────────────────────────────────────────
   Total Positives:             2412

🔴 NEGATIVE SAMPLES:
   Hard Negatives (FP IT2):      263
   Hard Negatives (FP IT1):      340
   Easy Negatives (Original):      0
   ────────────────────────────────────────
   Total Negatives:              603

⚖️  Negative:Positive Ratio:     0.250



✍️  Writing val: 100%|███████████████████████| 603/603 [00:00<00:00, 936.91it/s]



🔥 STARTING TRAINING
New https://pypi.org/project/ultralytics/8.4.4 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.10.19 torch-2.9.1+cu128 CUDA:0 (Tesla V100-PCIE-32GB, 32494MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/biopsy_gregorova/hpylori_project/yolo_it2_final/data.yaml, degrees=180, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/biopsy_gregorova/hpylori_p

Traceback (most recent call last):
  File "/home/biopsy_gregorova/miniconda3/envs/pyloribacteria/lib/python3.10/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/home/biopsy_gregorova/miniconda3/envs/pyloribacteria/lib/python3.10/multiprocessing/connection.py", line 177, in close
    self._close()
  File "/home/biopsy_gregorova/miniconda3/envs/pyloribacteria/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     43/150      4.84G      1.572      3.847      2.037          3        512: 100% ━━━━━━━━━━━━ 142/142 6.6it/s 21.6s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 6.4it/s 2.9s0.2s
                   all        596        788      0.412      0.368      0.326      0.137

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     44/150      4.84G      1.571      3.851      2.039          1        512: 100% ━━━━━━━━━━━━ 142/142 6.6it/s 21.4s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 6.3it/s 3.0s0.2s
                   all        596        788      0.412      0.348      0.321       0.13

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     45/150      4.84G      1.569      3.811      2.041          8        512

In [9]:
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import shutil
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict

# ====================================================================
# CONFIGURATION
# ====================================================================
class EvalConfig:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    # Point to the training output from Cell 1
    TRAIN_OUTPUT_DIR = PROJECT_ROOT / "yolo_it2_final"
    MODEL_PATH = TRAIN_OUTPUT_DIR / "train/weights/best.pt"
    
    # Output for this evaluation run
    EVAL_OUTPUT_DIR = TRAIN_OUTPUT_DIR / "evaluation_candidates"
    
    # Data Sources (Unverified Data)
    # 1. Remaining Test Data (Slides we haven't fully checked)
    TEST_DATA_FULL = PROJECT_ROOT / "master-data/separated_patches/test_data_full/images"
    # 2. Original "Easy" Negatives (Might contain missed bacteria)
    ORIGINAL_NEG_IMG = PROJECT_ROOT / "master-data/separated_patches/negative/images"
    
    # Inference Parameters
    BATCH_SIZE = 200
    CONF_THRESHOLD = 0.15  # Low threshold to catch everything for review
    IOU_THRESHOLD = 0.45
    
    def __init__(self):
        self.EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ====================================================================
# EVALUATOR
# ====================================================================
class CandidateMiner:
    def __init__(self, config: EvalConfig):
        self.cfg = config
        print(f"🔥 Loading Model: {config.MODEL_PATH}")
        self.model = YOLO(str(config.MODEL_PATH))
        self.excluded_files = self._get_training_files()
        
    def _get_training_files(self):
        """Don't re-evaluate images the model was trained on"""
        train_imgs = list((self.cfg.TRAIN_OUTPUT_DIR / "images/train").glob("*.png"))
        val_imgs = list((self.cfg.TRAIN_OUTPUT_DIR / "images/val").glob("*.png"))
        excluded = set(p.name for p in train_imgs + val_imgs)
        print(f"🛑 Excluding {len(excluded)} patches already used in training.")
        return excluded

    def _batch_predict(self, image_list, desc):
        """Run inference safely in batches"""
        detections = []
        
        for i in tqdm(range(0, len(image_list), self.cfg.BATCH_SIZE), desc=desc):
            batch = image_list[i : i + self.cfg.BATCH_SIZE]
            
            # Run YOLO
            results = self.model.predict(
                batch, 
                conf=self.cfg.CONF_THRESHOLD, 
                iou=self.cfg.IOU_THRESHOLD,
                verbose=False,
                device=0
            )
            
            # Process Results
            for img_path, result in zip(batch, results):
                if result.boxes and len(result.boxes) > 0:
                    # Found something!
                    max_conf = float(result.boxes.conf.max())
                    box_count = len(result.boxes)
                    
                    detections.append({
                        'path': img_path,
                        'max_conf': max_conf,
                        'count': box_count,
                        'boxes': result.boxes.xyxy.cpu().numpy()
                    })
                    
        return detections

    def run_mining_operation(self):
        print("\n🚀 STARTING CANDIDATE MINING FOR ITERATION 3")
        
        # 1. Collect all unverified images
        print("🔍 Scanning directories...")
        test_imgs = list(self.cfg.TEST_DATA_FULL.glob("*.png"))
        neg_imgs = list(self.cfg.ORIGINAL_NEG_IMG.glob("*.png"))
        
        all_candidates = test_imgs + neg_imgs
        
        # 2. Filter out training data
        clean_candidates = [p for p in all_candidates if p.name not in self.excluded_files]
        
        print(f"   Total Unverified Images: {len(clean_candidates):,}")
        
        if not clean_candidates:
            print("❌ No images to check.")
            return

        # 3. Run Inference
        print(f"\n⚡ Running Inference (Conf > {self.cfg.CONF_THRESHOLD})...")
        findings = self._batch_predict(clean_candidates, "Mining")
        
        # 4. Save & Report
        self._save_results(findings, len(clean_candidates))

    def _save_results(self, findings, total_scanned):
        print("\n" + "="*60)
        print("📊 MINING RESULTS")
        print("="*60)
        print(f"   Scanned:      {total_scanned:,}")
        print(f"   Found Candidates: {len(findings):,} ({(len(findings)/total_scanned):.1%})")
        
        if not findings:
            return

        # Sort by confidence (highest first) -> These are best for verification
        findings.sort(key=lambda x: x['max_conf'], reverse=True)
        
        # Save list for review tool / xml generation
        output_json = self.cfg.EVAL_OUTPUT_DIR / "candidates_for_verification.json"
        
        serializable_findings = []
        for f in findings:
            serializable_findings.append({
                'patch': f['path'].name,
                'path': str(f['path']),
                'max_conf': f['max_conf'],
                'count': f['count']
            })
            
        with open(output_json, 'w') as f:
            json.dump(serializable_findings, f, indent=2)
            
        print(f"💾 Candidate list saved: {output_json}")
        
        # Save Visualization of Top 20 Candidates
        vis_dir = self.cfg.EVAL_OUTPUT_DIR / "top_candidates_visuals"
        vis_dir.mkdir(exist_ok=True)
        
        print(f"🎨 Saving visuals for top 20 candidates...")
        for i, item in enumerate(findings[:20]):
            img = cv2.imread(str(item['path']))
            for box in item['boxes']:
                x1, y1, x2, y2 = map(int, box)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
            
            cv2.imwrite(str(vis_dir / f"rank_{i+1}_{item['path'].name}"), img)

if __name__ == "__main__":
    cfg = EvalConfig()
    if cfg.MODEL_PATH.exists():
        miner = CandidateMiner(cfg)
        miner.run_mining_operation()
    else:
        print("❌ Train model first!")

🔥 Loading Model: /home/biopsy_gregorova/hpylori_project/yolo_it2_final/train/weights/best.pt
🛑 Excluding 2781 patches already used in training.

🚀 STARTING CANDIDATE MINING FOR ITERATION 3
🔍 Scanning directories...
   Total Unverified Images: 137,335

⚡ Running Inference (Conf > 0.15)...


Mining: 100%|█████████████████████████████████| 687/687 [33:31<00:00,  2.93s/it]



📊 MINING RESULTS
   Scanned:      137,335
   Found Candidates: 3,135 (2.3%)
💾 Candidate list saved: /home/biopsy_gregorova/hpylori_project/yolo_it2_final/evaluation_candidates/candidates_for_verification.json
🎨 Saving visuals for top 20 candidates...


In [16]:
#!/usr/bin/env python3
"""
H. pylori Global XML Generator (Clean NMS Mode)
===============================================
1. Detects bacteria in patches.
2. Converts to Global Coordinates.
3. Removes duplicates using Global NMS (Non-Maximum Suppression).
4. Generates clean Aperio XMLs.
"""

import json
import re
import torch
import gc
import numpy as np
import xml.etree.ElementTree as ET
from xml.dom import minidom
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict

# ====================================================================
# CONFIGURATION
# ====================================================================
class Config:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    MODEL_PATH = PROJECT_ROOT / "yolo_it2_final/train/weights/best.pt"
    CANDIDATES_JSON = PROJECT_ROOT / "yolo_it2_final/evaluation_candidates/candidates_for_verification.json"
    TEST_IMAGES_DIR = PROJECT_ROOT / "master-data/separated_patches/test_data_full/images"
    OUTPUT_DIR = PROJECT_ROOT / "wsi_global_xmls_test_batch2_clean"
    
    CONF_THRESHOLD = 0.15 
    IOU_THRESHOLD = 0.45   # Local NMS (inside YOLO)
    
    # GLOBAL NMS SETTINGS (The Fix)
    GLOBAL_NMS_IOU = 0.20  # If two global boxes overlap by 20%, remove the lower confidence one
    
    MPP = "0.253200"
    LINE_COLOR = "65280"
    INFERENCE_BATCH_SIZE = 50 

# ====================================================================
# HELPER FUNCTIONS
# ====================================================================
def parse_filename(filename):
    match = re.search(r'(.+?)_x(\d+)_y(\d+)', filename.lower())
    if match:
        return match.group(1), int(match.group(2)), int(match.group(3))
    return filename, 0, 0

def prettify_xml(elem):
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="\t")

def py_cpu_nms(dets, thresh):
    """Pure Python NMS to merging global overlapping boxes."""
    if len(dets) == 0: return []
    
    x1 = dets[:, 0]
    y1 = dets[:, 1]
    x2 = dets[:, 2]
    y2 = dets[:, 3]
    scores = dets[:, 4]

    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]

    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])

        w = np.maximum(0.0, xx2 - xx1)
        h = np.maximum(0.0, yy2 - yy1)
        inter = w * h
        ovr = inter / (areas[i] + areas[order[1:]] - inter)

        inds = np.where(ovr <= thresh)[0]
        order = order[inds + 1]

    return dets[keep]

def save_aperio_xml(wsi_id, detections, output_dir):
    root = ET.Element("Annotations", MicronsPerPixel=Config.MPP)
    annotation = ET.SubElement(root, "Annotation", Id="1", Name="YOLO_Cleaned", 
                               ReadOnly="0", NameReadOnly="0", LineColorReadOnly="0", 
                               Incremental="0", Type="4", LineColor=Config.LINE_COLOR, 
                               Visible="1", Selected="1", MarkupImagePath="", MacroName="")
    ET.SubElement(annotation, "Attributes")
    regions = ET.SubElement(annotation, "Regions")
    ET.SubElement(regions, "RegionAttributeHeaders")

    # detections is [x_min, y_min, x_max, y_max, conf]
    for i, det in enumerate(detections):
        x_min, y_min, x_max, y_max, conf = det
        
        # Ensure integer coordinates for XML
        x_min, y_min, x_max, y_max = map(int, [x_min, y_min, x_max, y_max])

        region = ET.SubElement(regions, "Region", Id=str(i+1), Type="0", Zoom="1", 
                               Selected="0", ImageLocation="", ImageFocus="-1",
                               Length="0", Area="0", LengthMicrons="0", AreaMicrons="0",
                               Text=f"{conf:.2f}", NegativeROA="0", InputRegionId="0",
                               Analyze="1", DisplayId=str(i+1))
        ET.SubElement(region, "Attributes")
        vertices = ET.SubElement(region, "Vertices")
        
        # Clockwise vertices
        corners = [
            (x_min, y_min), (x_max, y_min), 
            (x_max, y_max), (x_min, y_max)
        ]
        for x, y in corners:
            ET.SubElement(vertices, "Vertex", X=str(x), Y=str(y), Z="0")

    output_path = output_dir / f"{wsi_id}.xml"
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(prettify_xml(root))

# ====================================================================
# MAIN LOGIC
# ====================================================================
def main():
    print(f"\n🚀 STARTING GLOBAL XML GENERATION (WITH NMS CLEANUP)")
    Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 1. Load Files
    target_files = []
    if Config.CANDIDATES_JSON.exists():
        print(f"📄 Loading candidates from: {Config.CANDIDATES_JSON.name}")
        with open(Config.CANDIDATES_JSON, 'r') as f:
            data = json.load(f)
            for item in data:
                path_str = item.get('path')
                if path_str and Path(path_str).exists():
                    target_files.append(Path(path_str))
                else:
                    rec_path = Config.TEST_IMAGES_DIR / item['patch']
                    if rec_path.exists(): target_files.append(rec_path)
        print(f"   ✅ Loaded {len(target_files)} candidate patches.")
    else:
        print("❌ Candidates JSON not found.")
        return

    # 2. Group by Slide
    print("🔍 Grouping by Slide ID...")
    slide_groups = defaultdict(list)
    for p in target_files:
        wsi_id, _, _ = parse_filename(p.name)
        slide_groups[wsi_id].append(p)
    print(f"   Processing {len(slide_groups)} slides.")

    # 3. Load Model
    print(f"🔥 Loading Model: {Config.MODEL_PATH}")
    model = YOLO(str(Config.MODEL_PATH))
    torch.cuda.empty_cache()
    gc.collect()
    
    # 4. Process Each Slide
    total_bacteria_raw = 0
    total_bacteria_clean = 0
    
    for wsi_id, patches in tqdm(slide_groups.items(), desc="Processing Slides"):
        slide_raw_detections = []
        
        # BATCH PROCESSING
        for i in range(0, len(patches), Config.INFERENCE_BATCH_SIZE):
            batch = patches[i : i + Config.INFERENCE_BATCH_SIZE]
            batch_paths = [str(p) for p in batch]
            
            results = model.predict(
                source=batch_paths,
                conf=Config.CONF_THRESHOLD,
                iou=Config.IOU_THRESHOLD,
                verbose=False,
                stream=False
            )
            
            for result in results:
                path = Path(result.path)
                _, off_x, off_y = parse_filename(path.name)
                
                if result.boxes:
                    # Get boxes in XYXY format directly
                    boxes = result.boxes.xyxy.cpu().numpy()
                    confs = result.boxes.conf.cpu().numpy()
                    
                    for box, conf in zip(boxes, confs):
                        x1, y1, x2, y2 = box
                        # Convert to GLOBAL coordinates
                        slide_raw_detections.append([
                            x1 + off_x, 
                            y1 + off_y, 
                            x2 + off_x, 
                            y2 + off_y, 
                            conf
                        ])
            
            del results
            torch.cuda.empty_cache()
        
        # APPLY GLOBAL NMS
        if slide_raw_detections:
            raw_count = len(slide_raw_detections)
            total_bacteria_raw += raw_count
            
            # Convert to numpy array for NMS
            dets_array = np.array(slide_raw_detections)
            cleaned_dets = py_cpu_nms(dets_array, Config.GLOBAL_NMS_IOU)
            
            clean_count = len(cleaned_dets)
            total_bacteria_clean += clean_count
            
            # Save XML
            save_aperio_xml(wsi_id, cleaned_dets, Config.OUTPUT_DIR)
            
    print("\n" + "="*60)
    print("✅ GENERATION COMPLETE")
    print(f"   XMLs Saved to: {Config.OUTPUT_DIR}")
    print(f"   Raw Detections: {total_bacteria_raw:,}")
    print(f"   Clean Detections: {total_bacteria_clean:,}")
    print(f"   Removed Duplicates: {total_bacteria_raw - total_bacteria_clean:,}")

if __name__ == "__main__":
    main()


🚀 STARTING GLOBAL XML GENERATION (WITH NMS CLEANUP)
📄 Loading candidates from: candidates_for_verification.json
   ✅ Loaded 3135 candidate patches.
🔍 Grouping by Slide ID...
   Processing 22 slides.
🔥 Loading Model: /home/biopsy_gregorova/hpylori_project/yolo_it2_final/train/weights/best.pt


Processing Slides: 100%|████████████████████████| 22/22 [00:41<00:00,  1.90s/it]


✅ GENERATION COMPLETE
   XMLs Saved to: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch2_clean
   Raw Detections: 3,475
   Clean Detections: 3,161
   Removed Duplicates: 314


In [15]:
#!/usr/bin/env python3
"""
H. Pylori Candidate Visualizer (Safe Mode)
==========================================
Generates "Slide Summary" images showing detected bacteria for each WSI.
Fixed to prevent CUDA OOM by batching inference.
"""

import cv2
import numpy as np
import json
import math
import torch
import gc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
from collections import defaultdict

# ====================================================================
# CONFIGURATION
# ====================================================================
class Config:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    
    # 1. Model
    MODEL_PATH = PROJECT_ROOT / "yolo_it2_final/train/weights/best.pt"
    
    # 2. Input: Candidates from the mining step
    CANDIDATES_JSON = PROJECT_ROOT / "yolo_it2_final/evaluation_candidates/candidates_for_verification.json"
    TEST_IMAGES_DIR = PROJECT_ROOT / "master-data/separated_patches/test_data_full/images"
    
    # 3. Output
    OUTPUT_DIR = PROJECT_ROOT / "yolo_it2_final/candidate_visualizations_zoom"
    
    # 4. Settings
    VISUALIZATION_CONF = 0.20  # Higher threshold for visual check
    GRID_COLS = 4              
    MAX_PATCHES_PER_SLIDE = 48 # Top 48 candidates per slide
    
    # SAFETY: Process images in tiny batches to clear VRAM
    VIZ_BATCH_SIZE = 20

# ====================================================================
# VISUALIZER UTILS
# ====================================================================
def create_slide_montage(slide_id, patches_data, output_dir):
    """Creates a single large image grid summarizing the slide"""
    if not patches_data:
        return

    # Sort by confidence (highest first)
    patches_data.sort(key=lambda x: x['max_conf'], reverse=True)
    
    # Limit to max patches
    display_patches = patches_data[:Config.MAX_PATCHES_PER_SLIDE]
    
    n_patches = len(display_patches)
    cols = Config.GRID_COLS
    rows = math.ceil(n_patches / cols)
    
    # Resize for grid to 256x256 to save space
    thumb_size = 512
    
    # Canvas size
    canvas_w = cols * thumb_size
    canvas_h = rows * thumb_size
    canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)
    
    for idx, item in enumerate(display_patches):
        img = item['img']
        # Resize if needed
        if img.shape[0] != thumb_size:
            img = cv2.resize(img, (thumb_size, thumb_size))
            
        # Add Confidence Text
        label = f"Conf: {item['max_conf']:.2f}"
        cv2.rectangle(img, (0, 0), (100, 20), (0,0,0), -1)
        cv2.putText(img, label, (5, 15), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
        
        # Calculate position
        r = idx // cols
        c = idx % cols
        
        y_start = r * thumb_size
        x_start = c * thumb_size
        
        canvas[y_start:y_start+thumb_size, x_start:x_start+thumb_size] = img
        
    # Save
    out_file = output_dir / f"Slide_{slide_id}_Summary.jpg"
    cv2.imwrite(str(out_file), canvas)

# ====================================================================
# MAIN LOGIC
# ====================================================================
def main():
    print("\n🎨 STARTING CANDIDATE VISUALIZATION (SAFE MODE)")
    Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 1. Load Candidates
    if not Config.CANDIDATES_JSON.exists():
        print("❌ Candidates JSON not found. Run mining first.")
        return
        
    with open(Config.CANDIDATES_JSON, 'r') as f:
        raw_candidates = json.load(f)
        
    # 2. Group by Slide
    print(f"📄 Loaded {len(raw_candidates)} candidates.")
    slide_groups = defaultdict(list)
    
    for item in raw_candidates:
        path_str = item.get('path')
        if not path_str or not Path(path_str).exists():
            p = Config.TEST_IMAGES_DIR / item['patch']
        else:
            p = Path(path_str)
            
        if p.exists():
            slide_id = p.stem.split('_')[0]
            slide_groups[slide_id].append(p)
            
    print(f"🔍 Processing {len(slide_groups)} slides...")
    
    # 3. Load Model
    model = YOLO(str(Config.MODEL_PATH))
    
    # 4. Process Each Slide
    for slide_id, image_paths in tqdm(slide_groups.items(), desc="Generating Reports"):
        
        slide_patches_data = []
        
        # BATCHING LOOP to prevent OOM
        for i in range(0, len(image_paths), Config.VIZ_BATCH_SIZE):
            batch_paths = image_paths[i : i + Config.VIZ_BATCH_SIZE]
            
            # Clean GPU before inference
            torch.cuda.empty_cache()
            gc.collect()
            
            # Run inference on small batch
            results = model.predict(
                source=[str(p) for p in batch_paths],
                conf=Config.VISUALIZATION_CONF,
                verbose=False,
                stream=False # Safer for small batches
            )
            
            for res, img_path in zip(results, batch_paths):
                if res.boxes and len(res.boxes) > 0:
                    # Load image (CPU memory)
                    orig_img = cv2.imread(str(img_path))
                    
                    max_conf = 0.0
                    for box in res.boxes:
                        x1, y1, x2, y2 = map(int, box.xyxy[0].cpu().numpy())
                        conf = float(box.conf[0].cpu().numpy())
                        max_conf = max(max_conf, conf)
                        
                        # Draw Box
                        cv2.rectangle(orig_img, (x1, y1), (x2, y2), (0, 0, 255), 2)
                    
                    slide_patches_data.append({
                        'img': orig_img,
                        'max_conf': max_conf,
                        'path': img_path
                    })
            
            # Explicit deletion
            del results
        
        # Create Montage for this slide
        if slide_patches_data:
            create_slide_montage(slide_id, slide_patches_data, Config.OUTPUT_DIR)
            
    print("\n" + "="*60)
    print("✅ VISUALIZATION COMPLETE")
    print(f"   📂 Output: {Config.OUTPUT_DIR}")
    print("   Check the 'Slide_XXXX_Summary.jpg' files.")

if __name__ == "__main__":
    main()


🎨 STARTING CANDIDATE VISUALIZATION (SAFE MODE)
📄 Loaded 3135 candidates.
🔍 Processing 22 slides...


Generating Reports: 100%|███████████████████████| 22/22 [01:41<00:00,  4.63s/it]


✅ VISUALIZATION COMPLETE
   📂 Output: /home/biopsy_gregorova/hpylori_project/yolo_it2_final/candidate_visualizations_zoom
   Check the 'Slide_XXXX_Summary.jpg' files.


In [17]:
#!/usr/bin/env python3
"""
H. Pylori XML Analysis Report
=============================
Parses generated Aperio XML files to produce detailed statistics
about detection counts, confidence distributions, and slide-level metrics.
"""

import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

# ====================================================================
# CONFIGURATION
# ====================================================================
class Config:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    
    # Input: Directory containing the .xml files you just generated
    XML_DIR = PROJECT_ROOT / "wsi_global_xmls_test_batch2_clean"
    
    # Output: Where to save the report/plots
    OUTPUT_DIR = XML_DIR / "analysis_report"
    
    # Reporting Thresholds
    CONF_BINS = [0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

# ====================================================================
# ANALYZER
# ====================================================================
def parse_xml_stats(xml_path):
    """Extracts confidence scores and count from a single XML file."""
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        confs = []
        for region in root.findall('.//Region'):
            # The 'Text' attribute holds the confidence score in our schema
            txt = region.get('Text')
            if txt:
                try:
                    conf = float(txt)
                    confs.append(conf)
                except ValueError:
                    pass # Handle non-numeric text if any
        return confs
    except ET.ParseError:
        print(f"⚠️ Error parsing {xml_path.name}")
        return []

def generate_report():
    print(f"\n📊 STARTING XML ANALYSIS")
    print(f"   Source: {Config.XML_DIR}")
    
    Config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    xml_files = list(Config.XML_DIR.glob("*.xml"))
    if not xml_files:
        print("❌ No XML files found!")
        return

    slide_stats = []
    all_confs = []
    
    print(f"   Found {len(xml_files)} slides. Parsing...")
    
    # 1. Parse Data
    for xml_file in tqdm(xml_files):
        confs = parse_xml_stats(xml_file)
        all_confs.extend(confs)
        
        if confs:
            slide_stats.append({
                'Slide ID': xml_file.stem,
                'Count': len(confs),
                'Max Conf': max(confs),
                'Mean Conf': np.mean(confs),
                'Min Conf': min(confs)
            })
        else:
             slide_stats.append({
                'Slide ID': xml_file.stem,
                'Count': 0, 'Max Conf': 0, 'Mean Conf': 0, 'Min Conf': 0
            })

    # Convert to DataFrame for easy handling
    df = pd.DataFrame(slide_stats)
    df = df.sort_values(by='Count', ascending=False)
    
    # 2. Print Global Stats
    total_detections = len(all_confs)
    print("\n" + "="*80)
    print("GLOBAL DETECTION STATISTICS")
    print("="*80)
    print(f"  Total Slides:      {len(xml_files)}")
    print(f"  Total Detections:  {total_detections:,}")
    if total_detections > 0:
        print(f"  Avg Detections/Slide: {total_detections/len(xml_files):.1f}")
        print(f"  Global Max Conf:   {max(all_confs):.4f}")
        print(f"  Global Mean Conf:  {np.mean(all_confs):.4f}")
    
    print("\n" + "-"*80)
    print("CONFIDENCE DISTRIBUTION")
    print("-" * 80)
    if all_confs:
        for thresh in Config.CONF_BINS:
            count = sum(1 for c in all_confs if c >= thresh)
            pct = (count / total_detections) * 100
            print(f"  ≥ {thresh:.2f}: {count:6,d} ({pct:5.1f}%)")

    # 3. Print Slide-Level Table
    print("\n" + "-"*80)
    print(f"{'SLIDE ID':<15} | {'COUNT':<8} | {'MAX CONF':<10} | {'MEAN CONF':<10}")
    print("-" * 80)
    for _, row in df.head(25).iterrows():
        print(f"{row['Slide ID']:<15} | {row['Count']:<8} | {row['Max Conf']:<10.4f} | {row['Mean Conf']:<10.4f}")
    if len(df) > 25:
        print(f"... and {len(df)-25} more slides.")

    # 4. Generate Visualizations
    print(f"\n🎨 Generating plots in {Config.OUTPUT_DIR}...")
    
    # Plot A: Detections per Slide
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df.head(20), x='Slide ID', y='Count', color='royalblue')
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 20 Slides by Bacteria Count')
    plt.tight_layout()
    plt.savefig(Config.OUTPUT_DIR / "detections_per_slide.png")
    plt.close()
    
    # Plot B: Confidence Histogram
    plt.figure(figsize=(10, 6))
    plt.hist(all_confs, bins=50, color='crimson', alpha=0.7, edgecolor='black')
    plt.xlabel('Confidence Score')
    plt.ylabel('Number of Detections')
    plt.title('Global Confidence Distribution (Recovered Candidates)')
    plt.grid(axis='y', alpha=0.3)
    plt.savefig(Config.OUTPUT_DIR / "confidence_histogram.png")
    plt.close()

    # 5. Save CSV
    csv_path = Config.OUTPUT_DIR / "slide_statistics.csv"
    df.to_csv(csv_path, index=False)
    print(f"✅ CSV Report saved: {csv_path}")

if __name__ == "__main__":
    generate_report()


📊 STARTING XML ANALYSIS
   Source: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch2_clean
   Found 22 slides. Parsing...


100%|██████████████████████████████████████████| 22/22 [00:00<00:00, 494.27it/s]


GLOBAL DETECTION STATISTICS
  Total Slides:      22
  Total Detections:  3,161
  Avg Detections/Slide: 143.7
  Global Max Conf:   0.7100
  Global Mean Conf:  0.2435

--------------------------------------------------------------------------------
CONFIDENCE DISTRIBUTION
--------------------------------------------------------------------------------
  ≥ 0.15:  3,161 (100.0%)
  ≥ 0.20:  2,030 ( 64.2%)
  ≥ 0.25:  1,293 ( 40.9%)
  ≥ 0.30:    748 ( 23.7%)
  ≥ 0.40:    172 (  5.4%)
  ≥ 0.50:     20 (  0.6%)
  ≥ 0.60:      3 (  0.1%)
  ≥ 0.70:      1 (  0.0%)
  ≥ 0.80:      0 (  0.0%)
  ≥ 0.90:      0 (  0.0%)

--------------------------------------------------------------------------------
SLIDE ID        | COUNT    | MAX CONF   | MEAN CONF 
--------------------------------------------------------------------------------
522934          | 809      | 0.5600     | 0.2726    
593440          | 537      | 0.5700     | 0.2369    
593452          | 299      | 0.5000     | 0.2208    
593438      

✅ CSV Report saved: /home/biopsy_gregorova/hpylori_project/wsi_global_xmls_test_batch2_clean/analysis_report/slide_statistics.csv


In [1]:
# ==============================================================================
# 📊 PAPER METRICS: THRESHOLD SENSITIVITY ANALYSIS
# ==============================================================================
# Purpose: Generate the final Precision vs. Threshold graph for the research paper.
# Logic: Re-evaluates Iteration 2 predictions against Iteration 3 verified ground truth.
# ==============================================================================

import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# 1. EXACT PATHS (Verified from your logs)
# ------------------------------------------------------------------------------
BASE_DIR = Path("/home/biopsy_gregorova/hpylori_project")

# The Candidates: Generated by THIS notebook (It2) previously
CANDIDATES_PATH = BASE_DIR / "yolo_it2_final/evaluation_candidates/candidates_for_verification.json"

# The Ground Truth: Generated by the NEXT notebook (It3 Verification)
# (Source: train_yolo_it3.ipynb, Cell 2 Output)
VERIFIED_PATH = BASE_DIR / "wsi_global_xmls_test_batch/verified_comparison_results3/reports/true_positive_patches.json"

# Output for Graphs
OUTPUT_DIR = BASE_DIR / "yolo_it2_final/paper_graphs_and_stats"

def generate_paper_metrics():
    print(f"🚀 STARTING THRESHOLD ANALYSIS")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 2. LOAD DATA
    # --------------------------------------------------------------------------
    if not CANDIDATES_PATH.exists():
        print(f"❌ ERROR: Candidates file not found at: {CANDIDATES_PATH}")
        return
    
    if not VERIFIED_PATH.exists():
        print(f"❌ ERROR: Verified file not found at: {VERIFIED_PATH}")
        print("   (Have you run the verification in train_yolo_it3.ipynb yet?)")
        return

    print("📂 Loading files...")
    with open(CANDIDATES_PATH, 'r') as f:
        candidates = json.load(f)
    with open(VERIFIED_PATH, 'r') as f:
        verified_data = json.load(f)
        
    # Create a set of verified filenames (ignoring paths to be safe)
    verified_names = {Path(item.get('patch', item.get('path', ''))).name for item in verified_data}
    
    print(f"   - Total Candidates (It2): {len(candidates)}")
    print(f"   - Total Verified Positives (It3): {len(verified_names)}")

    # 3. BUILD DATAFRAME
    # --------------------------------------------------------------------------
    data = []
    for item in candidates:
        fname = Path(item.get('path', item.get('patch', ''))).name
        conf = item.get('max_conf', 0.0)
        is_verified = fname in verified_names
        data.append({'filename': fname, 'confidence': conf, 'verified': is_verified})
        
    df = pd.DataFrame(data)

    # 4. CALCULATE METRICS AT EACH THRESHOLD
    # --------------------------------------------------------------------------
    results = []
    # Test thresholds from 0.15 up to 0.95
    thresholds = np.arange(0.15, 0.96, 0.05)
    
    for t in thresholds:
        subset = df[df['confidence'] >= t]
        sent_count = len(subset)
        correct_count = subset['verified'].sum()
        
        # Precision: % of sent patches that were actually bacteria
        precision = (correct_count / sent_count * 100) if sent_count > 0 else 0.0
        
        # Recall/Yield: How many of the 197 confirmed bacteria would we have found?
        total_true_positives = df['verified'].sum() # Should be 197
        recall_yield = (correct_count / total_true_positives * 100) if total_true_positives > 0 else 0.0
        
        results.append({
            'Threshold': t,
            'Sent_Count': sent_count,
            'Verified_Yield': correct_count,
            'Precision': precision,
            'Recall_Yield': recall_yield
        })
        
    metrics_df = pd.DataFrame(results)

    # 5. GENERATE PLOTS
    # --------------------------------------------------------------------------
    sns.set_style("whitegrid")
    
    # GRAPH 1: The "Paper" Graph (Precision vs Recall Trade-off)
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    # Precision Line (Blue)
    sns.lineplot(data=metrics_df, x='Threshold', y='Precision', color='blue', marker='o', label='Precision (%)', ax=ax1)
    ax1.set_ylabel('Precision (%)', color='blue', fontsize=12)
    ax1.set_xlabel('Confidence Threshold', fontsize=12)
    ax1.set_ylim(0, 105)
    
    # Yield Line (Green - Dashed)
    ax2 = ax1.twinx()
    sns.lineplot(data=metrics_df, x='Threshold', y='Recall_Yield', color='green', marker='x', linestyle='--', label='Recall (Yield) %', ax=ax2)
    ax2.set_ylabel('Recall / Yield (%)', color='green', fontsize=12)
    ax2.set_ylim(0, 105)
    
    plt.title('Impact of Threshold on Precision vs. Yield', fontsize=14)
    
    # Add a vertical line at 0.15 (Our chosen threshold)
    plt.axvline(x=0.15, color='gray', linestyle=':', label='Actual Threshold (0.15)')
    
    # Combine legends
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='center right')
    
    graph_path = OUTPUT_DIR / "precision_recall_tradeoff.png"
    plt.savefig(graph_path, dpi=300)
    plt.close()
    
    # 6. SAVE CSV
    # --------------------------------------------------------------------------
    csv_path = OUTPUT_DIR / "threshold_metrics.csv"
    metrics_df.to_csv(csv_path, index=False, float_format="%.4f")
    
    print("\n" + "="*60)
    print("✅ ANALYSIS RESULTS (Use this for your paper)")
    print("="*60)
    print(metrics_df[['Threshold', 'Sent_Count', 'Verified_Yield', 'Precision', 'Recall_Yield']].to_string(index=False, float_format="%.2f"))
    print(f"\n📂 Graph saved to: {graph_path}")
    print(f"📂 Data saved to:  {csv_path}")

if __name__ == "__main__":
    generate_paper_metrics()

🚀 STARTING THRESHOLD ANALYSIS
📂 Loading files...
   - Total Candidates (It2): 3135
   - Total Verified Positives (It3): 197

✅ ANALYSIS RESULTS (Use this for your paper)
 Threshold  Sent_Count  Verified_Yield  Precision  Recall_Yield
      0.15        3135             159       5.07        100.00
      0.20        1916             113       5.90         71.07
      0.25        1226              69       5.63         43.40
      0.30         700              45       6.43         28.30
      0.35         348              27       7.76         16.98
      0.40         158              10       6.33          6.29
      0.45          61               6       9.84          3.77
      0.50          18               4      22.22          2.52
      0.55           6               2      33.33          1.26
      0.60           3               1      33.33          0.63
      0.65           1               0       0.00          0.00
      0.70           1               0       0.00          0.0